# Translation Network (Method 1, Stage 3) — Kaggle Notebook

Trains the latent translation network: takes VAE2's encoding of a degraded image plus its damage mask, and learns to translate it toward VAE1's real-photo latent statistics -- the actual repair mechanism of Method 1. **Needs finished VAE1 and VAE2 checkpoints as input** -- train those first in their own separate notebooks (they can run in parallel with each other, but this stage needs both finished before it can start).

Includes validation support, mask-weighted supervised loss (the same fix applied to Method 3/Stage 1/VAE2 -- here the pixel-resolution mask is resized down to match the latent's spatial size before weighting), four separate damage-type folders, and real error propagation via `subprocess.run(...).check_returncode()` rather than `!python`.

**Before running anything:**
1. Right sidebar: **Settings > Accelerator > GPU T4 x2**
2. Right sidebar: **Settings > Internet > On**


## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


CUDA available: True
GPU: Tesla T4


## 2. Install dependencies

In [2]:
!pip install -q pillow tqdm opencv-python-headless scikit-image scipy pandas
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


torch 2.10.0+cu128
torchvision 0.25.0+cu128


## 3. Recreate the project files

Includes the validation-enabled, mask-weighted-loss training script and the tested VAE1/VAE2/translation-net architectures.


In [3]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [4]:
%%writefile models/__init__.py



Writing models/__init__.py


In [5]:
%%writefile models/blocks.py
"""
Basic building blocks shared by the VAE encoder and decoder.

These are standard, well-known layer patterns (residual blocks, strided
conv downsampling, transposed-conv upsampling) -- not specific to any one
paper's architecture. The VAE class that assembles them into the actual
"Bringing Old Photos Back to Life"-style domain VAE is in vae.py, and that
assembly (encoder depth, bottleneck design, how mu/logvar are produced) is
the part you're implementing yourself from the paper's description.
"""

import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    """A standard two-conv residual block with instance normalization.
    Used inside the encoder/decoder to add capacity without changing
    spatial resolution."""

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class DownsampleBlock(nn.Module):
    """Strided conv that halves spatial resolution and doubles channels
    (up to a cap), used to build the encoder's downsampling path."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    """Transposed conv that doubles spatial resolution, used to build the
    decoder's upsampling path (mirrors DownsampleBlock)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


Writing models/blocks.py


In [6]:
%%writefile models/vae.py
"""
A convolutional VAE with a *spatial* latent bottleneck (a small feature map,
not a single flattened vector) rather than the more familiar
flatten-to-a-vector VAE design.

Why spatial: "Bringing Old Photos Back to Life" needs the latent
representation to preserve rough spatial layout, so that later (in the
mapping network you'll build next) a damage mask can be used to tell the
model *where* in the latent space to focus repair. A flattened-vector
latent would throw that spatial correspondence away.

This same class is used for both VAE1 (domain A: real old photos) and VAE2
(domain B: clean photos) -- you'll instantiate two separate copies with
their own weights, one per domain, trained independently in
train_vae_domain_a.py / train_vae_domain_b.py.
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock, DownsampleBlock, UpsampleBlock


class Encoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Initial conv, no downsampling yet
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, base_channels, kernel_size=7, padding=0),
            nn.InstanceNorm2d(base_channels, affine=True),
            nn.ReLU(inplace=True),
        ]

        # Downsampling path: halve spatial resolution each step, double
        # channels up to max_channels
        channels = base_channels
        for _ in range(n_downsample):
            next_channels = min(channels * 2, max_channels)
            layers.append(DownsampleBlock(channels, next_channels))
            channels = next_channels

        # Residual blocks at the bottleneck resolution, adding capacity
        # without further downsampling
        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        self.backbone = nn.Sequential(*layers)

        # Separate 1x1 convs producing the mean and log-variance maps of
        # the latent distribution -- same spatial size as the backbone
        # output, just a different channel count
        self.to_mu = nn.Conv2d(channels, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(channels, latent_channels, kernel_size=1)

        self.bottleneck_channels = channels

    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        mu = self.to_mu(features)
        logvar = self.to_logvar(features)
        # Clamped here, at the source, so every downstream consumer
        # (reparameterize AND the KL loss) sees the same safe value --
        # without this, logvar can drift to a large value early in
        # training (before the encoder has learned sensible statistics),
        # and exp(logvar) genuinely overflows to inf, which corrupts
        # weights via the resulting huge gradient and collapses training
        # to NaN within the first few dozen batches. [-10, 10] is a
        # standard, well-established safe range: std = exp(0.5*logvar)
        # spans roughly 0.0067 to 148, wide enough to not meaningfully
        # constrain what the model can represent.
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, out_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Figure out the bottleneck channel count the same way the
        # encoder did, so the shapes line up
        channels = base_channels
        for _ in range(n_downsample):
            channels = min(channels * 2, max_channels)

        layers = [nn.Conv2d(latent_channels, channels, kernel_size=1)]

        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        # Upsampling path: mirror of the encoder's downsampling path
        channel_sequence = []
        c = base_channels
        for _ in range(n_downsample):
            channel_sequence.append(min(c * 2, max_channels))
            c = min(c * 2, max_channels)
        channel_sequence = [base_channels] + channel_sequence
        # channel_sequence e.g. [64, 128, 256, 512] for n_downsample=3;
        # we walk it backwards to go from bottleneck back to base_channels
        for i in range(n_downsample):
            in_ch = channel_sequence[n_downsample - i]
            out_ch = channel_sequence[n_downsample - i - 1]
            layers.append(UpsampleBlock(in_ch, out_ch))

        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(base_channels, out_channels, kernel_size=7, padding=0),
            nn.Tanh(),  # output in [-1, 1], matches how we'll normalize images
        ]

        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class DomainVAE(nn.Module):
    """
    Full VAE: encode -> reparameterize -> decode.

    Instantiate one of these per domain (real old photos / clean photos).
    The `Encoder`/`Decoder` above are shared *class* definitions but each
    DomainVAE instance gets its own independently-trained weights.
    """

    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)
        self.decoder = Decoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """The standard VAE reparameterization trick: sample z = mu + eps*std
        where eps ~ N(0, 1), so gradients can flow through the sampling step."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def vae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor,
             logvar: torch.Tensor, kl_weight: float = 1.0):
    """
    Standard VAE loss = reconstruction term + KL divergence term.

    Reconstruction uses L1 (tends to give sharper results than MSE for
    images -- this is a common choice in image-translation VAEs, not
    something unique to this paper).

    KL divergence pulls the latent distribution toward a standard normal,
    which is what makes the latent space smooth/well-structured enough for
    the mapping network to later translate between domains.
    """
    recon_loss = torch.nn.functional.l1_loss(recon, target)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + kl_weight * kl_loss
    return total_loss, recon_loss, kl_loss


def vae2_loss(recon_clean, recon_degraded, clean_target, mu_clean, logvar_clean,
              mu_degraded, logvar_degraded, kl_weight: float = 1.0, consistency_weight: float = 1.0):
    """
    VAE2's training objective is different from VAE1's plain reconstruction:
    it needs to learn a latent space where a clean photo AND a synthetically
    degraded version of it land close together, and BOTH decode back to the
    clean image. This is what lets the translation network later map a
    repaired latent through VAE2's decoder and get a clean-looking result.

    Four terms:
      1. Standard reconstruction of the clean branch (clean in -> clean out)
      2. Cross-reconstruction of the degraded branch (degraded in -> CLEAN
         out, not degraded out) -- this is the key difference from a plain
         autoencoder; it directly teaches "decode toward clean" regardless
         of which branch encoded the input
      3. Latent consistency: pulls the degraded branch's latent toward the
         clean branch's latent (clean side detached, so gradient flows
         into fixing the degraded encoder rather than both sides drifting
         together into a degenerate shortcut)
      4. KL divergence on both branches (standard VAE regularization)
    """
def vae2_loss(recon_clean, recon_degraded, clean_target, mu_clean, logvar_clean,
              mu_degraded, logvar_degraded, kl_weight: float = 1.0, consistency_weight: float = 1.0,
              mask=None, damage_weight: float = 5.0):
    """
    VAE2's training objective is different from VAE1's plain reconstruction:
    it needs to learn a latent space where a clean photo AND a synthetically
    degraded version of it land close together, and BOTH decode back to the
    clean image. This is what lets the translation network later map a
    repaired latent through VAE2's decoder and get a clean-looking result.

    Four terms:
      1. Standard reconstruction of the clean branch (clean in -> clean out)
      2. Cross-reconstruction of the degraded branch (degraded in -> CLEAN
         out, not degraded out) -- this is the key difference from a plain
         autoencoder; it directly teaches "decode toward clean" regardless
         of which branch encoded the input
      3. Latent consistency: pulls the degraded branch's latent toward the
         clean branch's latent (clean side detached, so gradient flows
         into fixing the degraded encoder rather than both sides drifting
         together into a degenerate shortcut)
      4. KL divergence on both branches (standard VAE regularization)

    If `mask` is provided (convention: 1.0 = clean, 0.0 = fully damaged),
    the degraded-branch reconstruction term (#2) is weighted to emphasize
    damaged pixels via `damage_weight` when computing `total_loss` (the
    value actually used for backward()) -- the same fix applied to Method
    3 and Method 2's Stage 1, and for the same reason: damaged pixels are
    typically a small fraction of the image, so this term can improve
    steadily just from the network getting better at reproducing the
    overall image, without specifically learning to repair damage. Only
    this term is weighted -- recon_loss_clean (#1) has no damage in its
    input/target pair at all, so there's nothing to weight there.

    The RETURNED recon_loss_degraded stays the plain, unweighted L1 value
    (comparable to pre-fix runs and to what evaluate.py measures) -- the
    weighted version is used only internally when building total_loss.
    """
    recon_loss_clean = torch.nn.functional.l1_loss(recon_clean, clean_target)
    recon_loss_degraded = torch.nn.functional.l1_loss(recon_degraded, clean_target)

    if mask is None:
        weighted_recon_degraded = recon_loss_degraded
    else:
        per_pixel_l1 = torch.abs(recon_degraded - clean_target)
        weight_map = 1.0 + damage_weight * (1.0 - mask)
        weighted_recon_degraded = (per_pixel_l1 * weight_map).mean()

    consistency_loss = torch.nn.functional.l1_loss(mu_degraded, mu_clean.detach())

    kl_clean = -0.5 * torch.mean(1 + logvar_clean - mu_clean.pow(2) - logvar_clean.exp())
    kl_degraded = -0.5 * torch.mean(1 + logvar_degraded - mu_degraded.pow(2) - logvar_degraded.exp())
    kl_loss = kl_clean + kl_degraded

    total_loss = (recon_loss_clean + weighted_recon_degraded
                  + consistency_weight * consistency_loss
                  + kl_weight * kl_loss)

    return total_loss, recon_loss_clean, recon_loss_degraded, consistency_loss, kl_loss


Writing models/vae.py


In [7]:
%%writefile models/translation_net.py
"""
The latent translation network -- this is where the actual "repair"
happens. Takes a latent map from either VAE1 (real damaged photo) or VAE2
(synthetic degraded photo), plus a damage mask resized to latent
resolution, and outputs a translated latent that VAE2's decoder can turn
into a clean-looking image.

Also defines the latent-space discriminator used for adversarial training
on real photos, where we have no ground-truth clean version to supervise
against directly (see train_translation_net.py for how the two training
signals -- supervised synthetic pairs + adversarial real photos -- combine).
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock


class LatentTranslationNet(nn.Module):
    def __init__(self, latent_channels: int = 64, n_residual_blocks: int = 6):
        super().__init__()

        # Fuse the latent map with the (resized) damage mask -- this is the
        # actual mask-conditioning mechanism: the mask is just concatenated
        # as an extra input channel, giving every residual block access to
        # "is this spatial location damaged" throughout the network.
        self.input_conv = nn.Sequential(
            nn.Conv2d(latent_channels + 1, latent_channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(latent_channels, affine=True),
            nn.ReLU(inplace=True),
        )

        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(latent_channels) for _ in range(n_residual_blocks)]
        )

        # Output projection back to latent space -- no activation, since
        # latent values (VAE means) aren't bounded to any fixed range
        self.output_conv = nn.Conv2d(latent_channels, latent_channels, kernel_size=3, padding=1)

    def forward(self, latent: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask arrives at image resolution; resize to match the latent
        # map's (much smaller) spatial size
        if mask.shape[-2:] != latent.shape[-2:]:
            mask = torch.nn.functional.interpolate(mask, size=latent.shape[-2:], mode="bilinear",
                                                     align_corners=False)
        x = torch.cat([latent, mask], dim=1)
        x = self.input_conv(x)
        x = self.residual_blocks(x)
        return self.output_conv(x)


class LatentDiscriminator(nn.Module):
    """PatchGAN-style discriminator operating directly on latent maps
    (not images). Outputs a spatial grid of real/fake scores rather than
    one global score, which gives a stronger, more localized training
    signal than a single scalar would."""

    def __init__(self, latent_channels: int = 64, base_channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(latent_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(base_channels * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(base_channels * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # No sigmoid -- used with BCEWithLogitsLoss for numerical stability
            nn.Conv2d(base_channels * 4, 1, kernel_size=3, padding=1),
        )

    def forward(self, latent: torch.Tensor) -> torch.Tensor:
        return self.net(latent)


class ImageDiscriminator(nn.Module):
    """PatchGAN-style discriminator operating on RGB images (not latents).

    Used to give the translator a genuine training signal tied to VAE1's
    *decoder*, which LatentDiscriminator alone never provides -- that one
    only ever supervises the translator's output against VAE2's encoder
    latent space. Real examples here are VAE1's own reconstructions of
    real old photos; fake examples are the translator's output decoded
    through VAE1's (frozen) decoder. See train_translation_net.py for how
    this combines with the existing latent-space adversarial loss."""

    def __init__(self, in_channels: int = 3, base_channels: int = 64):
        super().__init__()
        # spectral_norm caps each conv layer's Lipschitz constant to 1,
        # limiting how much the discriminator's output can change relative
        # to its input. This keeps it from becoming arbitrarily confident
        # too quickly -- which is what we saw happen without it: adv_d_img
        # collapsing toward 0.0000 within a handful of epochs regardless of
        # --image-adv-weight or --image-disc-lr, giving the generator
        # almost no useful gradient for the rest of training.
        sn = torch.nn.utils.parametrizations.spectral_norm
        self.net = nn.Sequential(
            sn(nn.Conv2d(in_channels, base_channels, kernel_size=4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),

            sn(nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)),
            nn.InstanceNorm2d(base_channels * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            sn(nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)),
            nn.InstanceNorm2d(base_channels * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            sn(nn.Conv2d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)),
            nn.InstanceNorm2d(base_channels * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # No sigmoid -- used with BCEWithLogitsLoss for numerical stability
            sn(nn.Conv2d(base_channels * 8, 1, kernel_size=3, padding=1)),
        )

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        return self.net(image)


Writing models/translation_net.py


In [8]:
%%writefile data/__init__.py



Writing data/__init__.py


In [9]:
%%writefile data/real_photo_dataset.py
"""
Dataset loader for VAE1 (domain A) training: real old photos, as collected
by loc_scraper.py / dpla_scraper.py. Unsupervised -- no labels needed, just
a folder of images.
"""

import os
import csv
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


class RealOldPhotoDataset(Dataset):
    """
    Expects the folder layout produced by loc_scraper.py / dpla_scraper.py:

        <root>/images/*.jpg
        <root>/manifest.csv   (optional, only used to list valid filenames)

    If manifest.csv is present, filenames are read from it (keeps you in
    sync with whatever passed your download-time validation). Otherwise
    falls back to globbing every image file in <root>/images/.
    """

    def __init__(self, root: str, image_size: int = 256, augment: bool = True):
        self.root = root
        self.images_dir = os.path.join(root, "images")
        self.image_size = image_size
        self.augment = augment

        self.filenames = self._load_filenames()
        if len(self.filenames) == 0:
            raise ValueError(f"No images found under {self.images_dir}")

        # Resize the short side up a bit past image_size so RandomCrop has
        # room to move -- this is a standard cheap augmentation for
        # unsupervised reconstruction training.
        load_size = int(image_size * 1.12)

        transform_list = [
            T.Resize(load_size),
        ]
        if augment:
            transform_list += [
                T.RandomCrop(image_size),
                T.RandomHorizontalFlip(),
            ]
        else:
            transform_list += [T.CenterCrop(image_size)]

        transform_list += [
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
        ]
        self.transform = T.Compose(transform_list)

    def _load_filenames(self):
        manifest_path = os.path.join(self.root, "manifest.csv")
        if os.path.exists(manifest_path):
            filenames = []
            with open(manifest_path, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    fn = row.get("local_filename")
                    if fn:
                        filenames.append(fn)
            if filenames:
                return filenames

        # fallback: glob the images directory directly
        if not os.path.isdir(self.images_dir):
            return []
        valid_ext = (".jpg", ".jpeg", ".png")
        return [f for f in os.listdir(self.images_dir) if f.lower().endswith(valid_ext)]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        path = os.path.join(self.images_dir, filename)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            # If something slipped past validation and is unreadable,
            # fall back to a random other item rather than crashing an
            # entire training run over one bad file.
            return self.__getitem__(random.randrange(len(self)))

        img_tensor = self.transform(img)
        return {"image": img_tensor, "filename": filename}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Inverse of the Normalize(mean=0.5, std=0.5) above, for saving/viewing
    reconstructed images. Maps [-1, 1] back to [0, 1]."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


Writing data/real_photo_dataset.py


In [10]:
%%writefile data/vae2_pair_dataset.py
"""
Dataset for VAE2 (domain B) and the latent translation network: yields
(clean, degraded, mask) triples. Degraded images are composited on the fly
using your FilmDamageSimulator masks (screen blend by default -- see
composite_damage.py), same approach as the DiffBIR Stage 1 dataset.

Returning the raw mask (not just the composited degraded image) is what's
new here -- VAE2 training itself doesn't need it, but the translation
network trained on top of VAE2 does, since it uses the mask to know where
to apply heavier repair.
"""

import os
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF


class VAE2PairDataset(Dataset):
    def __init__(self, clean_dir: str, masks_dir, image_size: int = 256, augment: bool = True,
                 blend_mode: str = "screen", white_probability: float = 0.8):
        if blend_mode not in ("screen", "multiply", "mixed"):
            raise ValueError(f"blend_mode must be 'screen', 'multiply', or 'mixed', got '{blend_mode}'")
        if not (0.0 <= white_probability <= 1.0):
            raise ValueError(f"white_probability must be between 0 and 1, got {white_probability}")
        self.clean_dir = clean_dir
        self.masks_dirs = [masks_dir] if isinstance(masks_dir, str) else list(masks_dir)
        self.image_size = image_size
        self.augment = augment
        self.blend_mode = blend_mode
        self.white_probability = white_probability

        valid_ext = (".jpg", ".jpeg", ".png")
        self.clean_files = [f for f in os.listdir(clean_dir) if f.lower().endswith(valid_ext)]

        self.mask_files = []
        for d in self.masks_dirs:
            for f in os.listdir(d):
                if f.lower().endswith(".png") and not f.startswith("binarised_mask"):
                    self.mask_files.append(os.path.join(d, f))

        if len(self.clean_files) == 0:
            raise ValueError(f"No clean images found in {clean_dir}")
        if len(self.mask_files) == 0:
            raise ValueError(f"No usable masks found in {self.masks_dirs}")

        load_size = int(image_size * 1.12)
        self.clean_resize = T.Resize(load_size)
        self.image_size_final = image_size

    def __len__(self):
        return len(self.clean_files)

    def _load_clean(self, idx):
        path = os.path.join(self.clean_dir, self.clean_files[idx])
        img = Image.open(path).convert("RGB")
        return self.clean_resize(img)

    def _load_random_mask(self):
        path = random.choice(self.mask_files)
        return Image.open(path).convert("L")

    def _synchronized_crop_and_flip(self, clean_img, mask_img):
        mask_img = mask_img.resize(clean_img.size, Image.BILINEAR)

        if self.augment:
            i, j, h, w = T.RandomCrop.get_params(clean_img, output_size=(self.image_size_final, self.image_size_final))
            clean_img = TF.crop(clean_img, i, j, h, w)
            mask_img = TF.crop(mask_img, i, j, h, w)
            if random.random() < 0.5:
                clean_img = TF.hflip(clean_img)
                mask_img = TF.hflip(mask_img)
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                mask_img = TF.rotate(mask_img, angle)
        else:
            clean_img = TF.center_crop(clean_img, (self.image_size_final, self.image_size_final))
            mask_img = TF.center_crop(mask_img, (self.image_size_final, self.image_size_final))

        return clean_img, mask_img

    def __getitem__(self, idx):
        try:
            clean_img = self._load_clean(idx)
            mask_img = self._load_random_mask()
        except Exception:
            return self.__getitem__(random.randrange(len(self)))

        clean_img, mask_img = self._synchronized_crop_and_flip(clean_img, mask_img)

        clean_tensor = TF.to_tensor(clean_img)
        mask_tensor = TF.to_tensor(mask_img)

        # In "mixed" mode, each sample independently rolls screen vs.
        # multiply according to white_probability -- see the note in
        # common/degraded_pair_dataset.py for the full reasoning.
        if self.blend_mode == "mixed":
            sample_blend = "screen" if random.random() < self.white_probability else "multiply"
        else:
            sample_blend = self.blend_mode

        if sample_blend == "screen":
            degraded_tensor = 1.0 - (1.0 - clean_tensor) * mask_tensor
        else:
            degraded_tensor = clean_tensor * mask_tensor

        normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        clean_tensor = normalize(clean_tensor)
        degraded_tensor = normalize(degraded_tensor)
        # mask stays in [0, 1] -- it's used as a conditioning signal, not an
        # image to reconstruct, so it doesn't need the [-1, 1] normalization

        return {"clean": clean_tensor, "degraded": degraded_tensor, "mask": mask_tensor}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    return (tensor * 0.5 + 0.5).clamp(0, 1)


Writing data/vae2_pair_dataset.py


In [11]:
%%writefile train_translation_net.py
"""
Train the latent translation network: the actual repair mechanism of this
method. Combines two training signals each step:

1. SUPERVISED (synthetic pairs, full ground truth available): encode a
   clean photo and its synthetically-degraded twin through frozen VAE2,
   translate the degraded latent with the mask, and directly supervise
   against the known clean latent with L1 loss. This teaches the network
   HOW to use the mask to repair damage.

2. ADVERSARIAL (real old photos, no ground truth): encode a real photo
   through frozen VAE1, translate it (no mask -- we don't know real
   damage locations), and train a discriminator to tell translated-real
   latents apart from genuine clean latents (from VAE2). The translation
   network is trained to fool it. This teaches the network to generalize
   the repair behavior learned from (1) to real-photo statistics, which
   is the actual domain-gap-closing step this whole method exists for.

Also includes an identity loss (translating an already-clean latent should
leave it roughly unchanged) which is a standard stabilizing trick borrowed
from CycleGAN-style unpaired translation training.

VAE1 and VAE2 are both loaded frozen from their own checkpoints and never
updated here -- only the translation network and discriminator train.

Usage:
    python train_translation_net.py \
        --vae1-checkpoint ./runs/vae_domain_a/checkpoints/vae_domain_a_epoch0050.pt \
        --vae2-checkpoint ./runs/vae_domain_b/checkpoints/vae_domain_b_epoch0050.pt \
        --real-photo-dir ./real_old_photos --clean-dir ./voc_data --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --out-dir ./runs/translation_net
"""

import argparse
import os
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE
from models.translation_net import LatentTranslationNet, LatentDiscriminator, ImageDiscriminator
from data.real_photo_dataset import RealOldPhotoDataset, denormalize as denorm_a
from data.vae2_pair_dataset import VAE2PairDataset, denormalize as denorm_b


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    def __init__(self, log_path):
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def load_frozen_vae(checkpoint_path, device):
    """Loads a VAE1 or VAE2 checkpoint, reconstructing the architecture
    from the checkpoint's own saved args (same pattern as evaluate.py) so
    there's no risk of mismatching --latent-channels etc. by hand.
    Freezes all parameters -- this VAE is used only for encoding/decoding,
    never updated during translation-network training."""
    ckpt = torch.load(checkpoint_path, map_location=device)
    train_args = ckpt.get("args", {})
    model = DomainVAE(
        in_channels=3,
        n_downsample=train_args.get("n_downsample", 3),
        n_residual_blocks=train_args.get("n_residual_blocks", 4),
        latent_channels=train_args.get("latent_channels", 64),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, train_args


def save_sample_grid(vae1, vae2, translator, real_batch, synth_batch, out_path, device, max_images=4):
    """Two rows of comparisons:
      - synthetic: degraded | translated->decoded | clean target (has ground truth)
      - real: real damaged photo | translated->decoded (no ground truth, this
        is the actual real-world use case the whole method targets)
    """
    translator.eval()
    with torch.no_grad():
        n = min(max_images, synth_batch["clean"].shape[0])
        clean = synth_batch["clean"][:n].to(device)
        degraded = synth_batch["degraded"][:n].to(device)
        mask = synth_batch["mask"][:n].to(device)

        mu_degraded, _ = vae2.encoder(degraded)
        translated_synth = translator(mu_degraded, mask)
        decoded_synth = vae2.decoder(translated_synth)

        n_real = min(max_images, real_batch["image"].shape[0])
        real = real_batch["image"][:n_real].to(device)
        mu_real, _ = vae1.encoder(real)
        zero_mask = torch.zeros((n_real, 1, real.shape[-2], real.shape[-1]), device=device)
        translated_real = translator(mu_real, zero_mask)
        decoded_real = vae2.decoder(translated_real)

        row1 = torch.cat([denorm_b(degraded), denorm_b(decoded_synth), denorm_b(clean)], dim=0)
        row2 = torch.cat([denorm_a(real), denorm_a(decoded_real)], dim=0)

        vutils.save_image(row1, out_path.replace(".png", "_synthetic.png"), nrow=n)
        vutils.save_image(row2, out_path.replace(".png", "_real.png"), nrow=n_real)
    translator.train()


def weighted_latent_l1_loss(pred, target, pixel_mask, damage_weight=5.0):
    """The supervised loss operates on LATENT tensors (much smaller spatial
    resolution than the original image, due to VAE2's encoder downsampling),
    while the damage mask is at full pixel resolution -- so unlike Method 3/
    Stage 1/VAE2's pixel-space losses, the mask must be resized DOWN to match
    the latent's spatial size before it can be used to weight the loss.

    Same underlying motivation as those other fixes: damaged latent
    positions are typically a small fraction of the total, so plain L1 can
    improve steadily from the translator getting better at latents overall,
    without specifically learning to use the mask to repair damage.

    Returns (weighted_loss_for_backprop, plain_l1_for_logging) -- the plain
    value stays comparable across runs and to damage_weight=0.
    """
    plain_l1 = torch.nn.functional.l1_loss(pred, target)

    if damage_weight == 0:
        return plain_l1, plain_l1

    # Resize the pixel-resolution mask down to the latent's spatial size.
    # 'bilinear' is appropriate here since mask values are continuous
    # ([0,1], not strictly binary) after the original antialiased compositing.
    resized_mask = F.interpolate(pixel_mask, size=pred.shape[-2:], mode="bilinear", align_corners=False)

    per_pixel_l1 = torch.abs(pred - target)
    weight_map = 1.0 + damage_weight * (1.0 - resized_mask)
    weighted_l1 = (per_pixel_l1 * weight_map).mean()

    return weighted_l1, plain_l1


@torch.no_grad()
def run_validation(vae2, translator, val_dataloader, device, damage_weight=5.0):
    """Only the supervised synthetic-pair loss is used for validation --
    it's the one component with genuine ground truth (the known clean
    latent) to check against. The adversarial and identity losses don't
    have a meaningful 'correct answer' on held-out data the way L1
    against a known target does, so they're not included here.

    Returns (weighted_loss, plain_l1) -- weighted_loss drives best.pt
    selection, plain_l1 stays comparable across runs."""
    translator.eval()
    total_weighted, total_plain, n_batches = 0.0, 0.0, 0
    for batch in val_dataloader:
        clean = batch["clean"].to(device, non_blocking=True)
        degraded = batch["degraded"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)

        mu_clean, _ = vae2.encoder(clean)
        mu_degraded, _ = vae2.encoder(degraded)
        translated = translator(mu_degraded, mask)
        weighted_loss, plain_l1 = weighted_latent_l1_loss(translated, mu_clean, mask, damage_weight=damage_weight)

        total_weighted += weighted_loss.item()
        total_plain += plain_l1.item()
        n_batches += 1
    translator.train()
    if n_batches == 0:
        return None, None
    return total_weighted / n_batches, total_plain / n_batches


def main():
    parser = argparse.ArgumentParser(description="Train the latent translation network.")
    parser.add_argument("--vae1-checkpoint", type=str, required=True)
    parser.add_argument("--vae2-checkpoint", type=str, required=True)
    parser.add_argument("--real-photo-dir", type=str, required=True)
    parser.add_argument("--clean-dir", type=str, required=True)
    parser.add_argument("--val-real-photo-dir", type=str, default=None,
                         help="optional held-out validation real-photo folder")
    parser.add_argument("--val-clean-dir", type=str, default=None,
                         help="optional held-out validation clean-photo folder. Both --val-real-photo-dir "
                              "and --val-clean-dir/--val-masks-dir must be set together to run validation.")
    parser.add_argument("--val-masks-dir", type=str, default=None, nargs="+")
    parser.add_argument("--masks-dir", type=str, required=True, nargs="+",
                         help="one or more mask folders; pass multiple to combine damage types kept "
                              "in separate folders, e.g. --masks-dir ./data/masks/scratches ./data/masks/smut")
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply", "mixed"], default="screen")
    parser.add_argument("--white-probability", type=float, default=0.8,
                         help="only used with --blend-mode mixed; fraction of samples using screen (white) blend")
    parser.add_argument("--damage-weight", type=float, default=5.0,
                         help="extra loss weight applied to damaged LATENT positions in the supervised "
                              "loss (the pixel-resolution mask is resized down to match the latent's "
                              "spatial size first). Without this, damaged positions are typically a small "
                              "fraction of the latent map, so the supervised loss can improve steadily "
                              "without the translator specifically learning to use the mask to repair "
                              "damage. Set to 0 to disable (plain unweighted L1, the old behavior).")
    parser.add_argument("--out-dir", type=str, default="./runs/translation_net")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--image-disc-lr", type=float, default=None,
                         help="learning rate for the image-space discriminator specifically, "
                              "separate from --lr. Defaults to --lr if unset. Useful for slowing "
                              "down a discriminator that's saturating (very low adv_d_img, high "
                              "adv_g_img) -- a lower rate here gives the generator more chance to "
                              "keep up, rather than the discriminator racing ahead every step.")
    parser.add_argument("--n-residual-blocks", type=int, default=6)
    parser.add_argument("--sup-weight", type=float, default=10.0,
                         help="weight on the supervised synthetic-pair loss")
    parser.add_argument("--identity-weight", type=float, default=1.0)
    parser.add_argument("--adv-weight", type=float, default=1.0)
    parser.add_argument("--image-adv-weight", type=float, default=1.0,
                         help="weight for the new image-space adversarial loss, which gives the "
                              "translator a genuine training signal tied to VAE1's decoder -- unlike "
                              "the existing latent-space adversarial loss, which only ever supervises "
                              "against VAE2's encoder latent space. Set to 0 to disable this branch "
                              "and fall back to the old latent-only behavior.")
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--steps-per-epoch", type=int, default=100,
                         help="both the real-photo and synthetic-pair dataloaders are capped to this many "
                              "batches per epoch, so they stay aligned regardless of underlying dataset size")
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--val-every", type=int, default=1,
                         help="run validation every N epochs (only used if both --val-real-photo-dir and "
                              "--val-clean-dir/--val-masks-dir are set)")
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--stop-after-epoch", type=int, default=1,
                         help="minimum epoch before early-stopping is even considered -- prevents "
                              "stopping on a lucky early epoch before training has genuinely converged. "
                              "Used together with --patience.")
    parser.add_argument("--patience", type=int, default=None,
                         help="if set, training stops early if validation loss hasn't improved on the "
                              "best-so-far value for this many CONSECUTIVE validation checks. Also "
                              "respects --stop-after-epoch. Leave unset to disable.")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")

    log(f"Loading frozen VAE1 from {args.vae1_checkpoint}")
    vae1, vae1_args = load_frozen_vae(args.vae1_checkpoint, device)
    log(f"Loading frozen VAE2 from {args.vae2_checkpoint}")
    vae2, vae2_args = load_frozen_vae(args.vae2_checkpoint, device)

    latent_channels = vae1_args.get("latent_channels", 64)
    assert latent_channels == vae2_args.get("latent_channels", 64), \
        "VAE1 and VAE2 must share the same --latent-channels -- they were trained with different values."

    log("Building datasets...")
    real_dataset = RealOldPhotoDataset(args.real_photo_dir, image_size=args.image_size, augment=True)
    synth_dataset = VAE2PairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size,
                                     augment=True, blend_mode=args.blend_mode, white_probability=args.white_probability)
    log(f"Real photos: {len(real_dataset)}. Synthetic pairs: {len(synth_dataset)} clean images, "
        f"{len(synth_dataset.mask_files)} masks.")

    real_sampler = RandomSampler(real_dataset, replacement=True,
                                  num_samples=args.steps_per_epoch * args.batch_size)
    synth_sampler = RandomSampler(synth_dataset, replacement=True,
                                   num_samples=args.steps_per_epoch * args.batch_size)
    real_loader = DataLoader(real_dataset, batch_size=args.batch_size, sampler=real_sampler,
                              num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                              persistent_workers=(args.num_workers > 0))
    synth_loader = DataLoader(synth_dataset, batch_size=args.batch_size, sampler=synth_sampler,
                               num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                               persistent_workers=(args.num_workers > 0))

    log("Fetching fixed sample batches for visualization...")
    fixed_real_batch = next(iter(real_loader))
    fixed_synth_batch = next(iter(synth_loader))
    log("Datasets ready.")

    val_dataloader = None
    if args.val_real_photo_dir and args.val_clean_dir:
        if not args.val_masks_dir:
            raise ValueError("--val-clean-dir requires --val-masks-dir")
        log(f"Building validation dataset from {args.val_clean_dir} / {args.val_masks_dir}...")
        val_dataset = VAE2PairDataset(args.val_clean_dir, args.val_masks_dir, image_size=args.image_size,
                                       augment=False, blend_mode=args.blend_mode, white_probability=args.white_probability)
        val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False,
                                     num_workers=args.num_workers, drop_last=False,
                                     pin_memory=(device.type == "cuda"))
        log(f"Loaded {len(val_dataset)} validation samples "
            f"(--val-real-photo-dir is accepted for symmetry with other args but not used here, "
            f"since validation is scored on the supervised synthetic pathway only)")

    best_val_loss = float("inf")
    epochs_since_improvement = 0

    translator = LatentTranslationNet(latent_channels=latent_channels,
                                       n_residual_blocks=args.n_residual_blocks).to(device)
    discriminator = LatentDiscriminator(latent_channels=latent_channels).to(device)
    image_discriminator = ImageDiscriminator(in_channels=3, base_channels=16).to(device)

    opt_g = torch.optim.Adam(translator.parameters(), lr=args.lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=args.lr, betas=(0.5, 0.999))
    image_disc_lr = args.image_disc_lr if args.image_disc_lr is not None else args.lr
    opt_d_img = torch.optim.Adam(image_discriminator.parameters(), lr=image_disc_lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        translator.load_state_dict(ckpt["translator_state_dict"])
        discriminator.load_state_dict(ckpt["discriminator_state_dict"])
        opt_g.load_state_dict(ckpt["opt_g_state_dict"])
        opt_d.load_state_dict(ckpt["opt_d_state_dict"])
        if "image_discriminator_state_dict" in ckpt:
            try:
                image_discriminator.load_state_dict(ckpt["image_discriminator_state_dict"])
                opt_d_img.load_state_dict(ckpt["opt_d_img_state_dict"])
            except RuntimeError as e:
                # The checkpoint has image-discriminator keys, but they don't
                # match this architecture -- e.g. resuming a pre-spectral-norm
                # checkpoint after adding spectral_norm, which renames every
                # conv layer's parameters. Starting fresh here is safe: the
                # translator and latent discriminator still resume correctly,
                # this only affects the image-space discriminator's own weights.
                log(f"  Image-discriminator checkpoint state doesn't match current "
                    f"architecture ({type(e).__name__}) -- starting it fresh instead "
                    f"of crashing. This is expected if you've changed its architecture "
                    f"(e.g. added spectral_norm) since this checkpoint was saved.")
        else:
            # Resuming from a checkpoint saved before the image-space
            # discriminator existed -- it starts fresh (randomly
            # initialized) rather than crashing on a missing key. This is
            # fine: the translator itself resumes correctly, and the new
            # discriminator just needs a few steps to start giving useful
            # signal, same as it would from scratch.
            log("  No image-discriminator state in checkpoint -- starting it fresh.")
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in translator.parameters())
    log(f"Translator has {num_params:,} parameters")
    log(f"Image discriminator learning rate: {image_disc_lr} (--lr is {args.lr})")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running = {"sup": 0.0, "identity": 0.0, "adv_g": 0.0, "adv_d": 0.0,
                   "adv_g_img": 0.0, "adv_d_img": 0.0}

        # mininterval=10 caps how often tqdm re-renders. Without this, Kaggle's
        # saved notebook output stores EVERY refresh as its own separate line
        # (no real terminal to overwrite in-place), which is what was making
        # notebook saves slow -- --log-every already gives real progress
        # visibility, so tqdm here is just a convenience display, not the
        # only source of feedback.
        progress_bar = tqdm(zip(real_loader, synth_loader), total=args.steps_per_epoch,
                             desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False,
                             mininterval=10)
        batch_end_time = time.time()

        for real_batch, synth_batch in progress_bar:
            data_time = time.time() - batch_end_time
            compute_start = time.time()

            real = real_batch["image"].to(device, non_blocking=True)
            clean = synth_batch["clean"].to(device, non_blocking=True)
            degraded = synth_batch["degraded"].to(device, non_blocking=True)
            mask = synth_batch["mask"].to(device, non_blocking=True)

            with torch.no_grad():
                mu_clean, _ = vae2.encoder(clean)
                mu_degraded, _ = vae2.encoder(degraded)
                mu_real, _ = vae1.encoder(real)

            zero_mask_real = torch.zeros((mu_real.shape[0], 1, real.shape[-2], real.shape[-1]), device=device)

            # ---- Train discriminator ----
            opt_d.zero_grad()
            translated_real_detached = translator(mu_real, zero_mask_real).detach()
            d_real_out = discriminator(mu_clean)
            d_fake_out = discriminator(translated_real_detached)
            loss_d_real = bce(d_real_out, torch.ones_like(d_real_out))
            loss_d_fake = bce(d_fake_out, torch.zeros_like(d_fake_out))
            loss_d = 0.5 * (loss_d_real + loss_d_fake)
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
            opt_d.step()

            # ---- Train image-space discriminator ----
            # Gives the translator a genuine training signal tied to VAE1's
            # decoder, unlike the latent-space discriminator above (which
            # only ever supervises against VAE2's encoder latent space).
            # Real examples: VAE1's own reconstruction of real old photos
            # (mu_real decoded through VAE1's own decoder -- what VAE1
            # itself considers a plausible real-photo-domain image). Fake
            # examples: the translator's output on real photos, decoded
            # through the same (frozen) VAE1 decoder.
            opt_d_img.zero_grad()
            with torch.no_grad():
                real_decoded = vae1.decoder(mu_real)
                fake_decoded_detached = vae1.decoder(translated_real_detached)
            d_img_real_out = image_discriminator(real_decoded)
            d_img_fake_out = image_discriminator(fake_decoded_detached)
            loss_d_img_real = bce(d_img_real_out, torch.ones_like(d_img_real_out))
            loss_d_img_fake = bce(d_img_fake_out, torch.zeros_like(d_img_fake_out))
            loss_d_img = 0.5 * (loss_d_img_real + loss_d_img_fake)
            loss_d_img.backward()
            torch.nn.utils.clip_grad_norm_(image_discriminator.parameters(), max_norm=1.0)
            opt_d_img.step()

            # ---- Train translator (generator) ----
            opt_g.zero_grad()

            translated_synth = translator(mu_degraded, mask)
            loss_sup_weighted, loss_sup = weighted_latent_l1_loss(translated_synth, mu_clean, mask,
                                                                   damage_weight=args.damage_weight)

            zero_mask_clean = torch.zeros_like(mask)
            translated_identity = translator(mu_clean, zero_mask_clean)
            loss_identity = torch.nn.functional.l1_loss(translated_identity, mu_clean)

            translated_real = translator(mu_real, zero_mask_real)
            d_out_for_g = discriminator(translated_real)
            loss_adv_g = bce(d_out_for_g, torch.ones_like(d_out_for_g))

            # VAE1's decoder is frozen (no optimizer updates its weights),
            # but its forward pass here is NOT under torch.no_grad() --
            # gradients need to flow through it back into the translator,
            # which is the whole point of this branch.
            fake_decoded = vae1.decoder(translated_real)
            d_img_out_for_g = image_discriminator(fake_decoded)
            loss_adv_g_img = bce(d_img_out_for_g, torch.ones_like(d_img_out_for_g))

            loss_g = (args.sup_weight * loss_sup_weighted
                      + args.identity_weight * loss_identity
                      + args.adv_weight * loss_adv_g
                      + args.image_adv_weight * loss_adv_g_img)
            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(translator.parameters(), max_norm=1.0)
            opt_g.step()

            compute_time = time.time() - compute_start

            running["sup"] += loss_sup.item()
            running["identity"] += loss_identity.item()
            running["adv_g"] += loss_adv_g.item()
            running["adv_d"] += loss_d.item()
            running["adv_g_img"] += loss_adv_g_img.item()
            running["adv_d_img"] += loss_d_img.item()
            global_step += 1

            progress_bar.set_postfix({"sup": f"{loss_sup.item():.4f}", "adv_d": f"{loss_d.item():.4f}"})

            if args.log_every and global_step % args.log_every == 0:
                log(f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s")

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_sample_grid(vae1, vae2, translator, fixed_real_batch, fixed_synth_batch,
                                  sample_path, device)
                log(f"  Saved sample grids: {sample_path.replace('.png', '_synthetic.png')} / "
                    f"{sample_path.replace('.png', '_real.png')}")

            batch_end_time = time.time()

        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] "
            f"sup={running['sup'] / args.steps_per_epoch:.4f} "
            f"identity={running['identity'] / args.steps_per_epoch:.4f} "
            f"adv_g={running['adv_g'] / args.steps_per_epoch:.4f} "
            f"adv_d={running['adv_d'] / args.steps_per_epoch:.4f} "
            f"adv_g_img={running['adv_g_img'] / args.steps_per_epoch:.4f} "
            f"adv_d_img={running['adv_d_img'] / args.steps_per_epoch:.4f} "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if val_dataloader is not None and epoch % args.val_every == 0:
            val_loss, val_plain_l1 = run_validation(vae2, translator, val_dataloader, device,
                                                      damage_weight=args.damage_weight)
            log(f"  [Validation] epoch {epoch}: weighted_supervised_loss={val_loss:.4f} plain_l1={val_plain_l1:.4f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_since_improvement = 0
                best_path = os.path.join(checkpoints_dir, "best.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "translator_state_dict": translator.state_dict(),
                    "discriminator_state_dict": discriminator.state_dict(),
                    "opt_g_state_dict": opt_g.state_dict(),
                    "opt_d_state_dict": opt_d.state_dict(),
                    "image_discriminator_state_dict": image_discriminator.state_dict(),
                    "opt_d_img_state_dict": opt_d_img.state_dict(),
                    "val_loss": val_loss,
                    "val_plain_l1": val_plain_l1,
                    "args": vars(args),
                }, best_path)
                log(f"  New best validation loss ({val_loss:.4f}) -- saved {best_path}")
            else:
                epochs_since_improvement += 1

            if (args.patience is not None and epoch >= args.stop_after_epoch
                    and epochs_since_improvement >= args.patience):
                log(f"  Validation loss hasn't improved on the best value ({best_val_loss:.4f}) for "
                    f"{epochs_since_improvement} consecutive checks (>= --patience {args.patience}) "
                    f"at epoch {epoch} -- stopping early.")
                ckpt_path = os.path.join(checkpoints_dir, f"translation_net_epoch{epoch:04d}.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "translator_state_dict": translator.state_dict(),
                    "discriminator_state_dict": discriminator.state_dict(),
                    "opt_g_state_dict": opt_g.state_dict(),
                    "opt_d_state_dict": opt_d.state_dict(),
                    "image_discriminator_state_dict": image_discriminator.state_dict(),
                    "opt_d_img_state_dict": opt_d_img.state_dict(),
                    "args": vars(args),
                }, ckpt_path)
                log(f"  Saved final checkpoint before stopping: {ckpt_path}")
                log(f"  Note: best.pt (val_loss={best_val_loss:.4f}) is likely more useful than this "
                    f"final checkpoint for downstream use, given training had stopped improving.")
                break

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"translation_net_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "global_step": global_step,
                "translator_state_dict": translator.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "opt_g_state_dict": opt_g.state_dict(),
                "opt_d_state_dict": opt_d.state_dict(),
                "image_discriminator_state_dict": image_discriminator.state_dict(),
                "opt_d_img_state_dict": opt_d_img.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    log(f"Training complete. Total time: {format_duration(time.time() - start_time)}")
    logger.close()


if __name__ == "__main__":
    main()


Writing train_translation_net.py


In [12]:
%%writefile composite_damage.py
"""
Composite a generated damage mask (from generate_synthetic_only.py or
damage_generator.py) onto a clean target image, producing a damaged/clean
training pair for restoration model training.

The mask convention from this codebase: 255 = clean/undamaged, values toward
0 = damaged (dust, dirt, scratches etc).

Three blend modes are supported:
  - "screen": LIGHTENS toward white at damaged pixels. This is the
    physically realistic choice for most scratch/abrasion damage, where the
    print's emulsion is scraped away and the lighter paper base shows
    through -- old photo scratches are usually bright/white marks, not dark
    ones.
  - "multiply": DARKENS toward black at damaged pixels. More appropriate for
    damage types that genuinely deposit dark material (soot/smut, heavy
    dirt, mold staining) rather than abrading the surface.
  - "mixed" (default): randomly picks screen or multiply for THIS composite,
    weighted by --white-probability (default 0.8 = 80% chance of white
    scratches-style damage, 20% chance of black smut-style damage). Since
    this tool composites one image at a time, running it repeatedly (e.g.
    in a loop over many images) with "mixed" will produce that ratio across
    the batch, rather than a single fixed appearance for every image.

Since a single generated mask can currently mix multiple damage types
(e.g. scratches + smut) without tracking which pixel came from which type,
blend selection here is a per-composite choice rather than automatic
per-pixel selection.

Usage:
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend multiply
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend mixed --white-probability 0.8
"""

import argparse
import random

import cv2 as cv
import numpy as np


def composite(clean_img, mask_img, blend="screen", white_probability=0.8):
    if clean_img.shape[:2] != mask_img.shape[:2]:
        mask_img = cv.resize(mask_img, (clean_img.shape[1], clean_img.shape[0]), interpolation=cv.INTER_LINEAR)

    mask_norm = mask_img.astype(np.float32) / 255.0
    if clean_img.ndim == 3 and mask_norm.ndim == 2:
        mask_norm = mask_norm[:, :, None]

    clean_f = clean_img.astype(np.float32)

    resolved_blend = blend
    if blend == "mixed":
        resolved_blend = "screen" if random.random() < white_probability else "multiply"

    if resolved_blend == "screen":
        # Lightens toward white at damaged (low-mask) pixels.
        damaged = 255.0 - (255.0 - clean_f) * mask_norm
    elif resolved_blend == "multiply":
        # Darkens toward black at damaged (low-mask) pixels.
        damaged = clean_f * mask_norm
    else:
        raise ValueError(f"Unknown blend mode '{blend}', expected 'screen', 'multiply', or 'mixed'")

    # Kept as a single return value (not a tuple) so existing callers that
    # do `damaged_img = composite(clean, mask)` keep working unchanged --
    # the resolved mode is attached as an attribute instead, for callers
    # that want to know which mode was actually picked in "mixed" mode.
    result = np.clip(damaged, 0, 255).astype(np.uint8)
    composite.last_resolved_blend = resolved_blend
    return result


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Composite a damage mask onto a clean image.')
    parser.add_argument('--clean', required=True, help='path to the clean input image')
    parser.add_argument('--mask', required=True, help='path to the generated grayscale damage mask')
    parser.add_argument('--out', required=True, help='path to write the damaged output image')
    parser.add_argument('--blend', choices=['screen', 'multiply', 'mixed'], default='screen',
                         help="'screen' (default) always produces light/white damage marks; "
                              "'multiply' always produces dark damage marks; "
                              "'mixed' randomly picks screen/multiply per --white-probability")
    parser.add_argument('--white-probability', type=float, default=0.8,
                         help="only used with --blend mixed; probability of picking screen (white) "
                              "over multiply (black) for this composite (default 0.8)")
    args = parser.parse_args()

    clean_img = cv.imread(args.clean, cv.IMREAD_UNCHANGED)
    mask_img = cv.imread(args.mask, cv.IMREAD_GRAYSCALE)

    damaged = composite(clean_img, mask_img, blend=args.blend, white_probability=args.white_probability)
    cv.imwrite(args.out, damaged)
    print(f"Wrote damaged image to {args.out} (blend={args.blend}, resolved to '{composite.last_resolved_blend}')")


Writing composite_damage.py


## 4. Attach your finished VAE1 and VAE2 checkpoints

Both must come from **Save Version → Create Dataset from Notebook Output** in your VAE1 and VAE2 notebooks respectively, then attached here via **Add Data**. Adjust both paths below to your actual attached dataset locations -- run `!ls /kaggle/input/` first if you're not sure of the exact names.


In [13]:
VAE1_CHECKPOINT = '/kaggle/input/notebooks/dorast/vae-a/runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0300.pt'  # adjust
VAE2_CHECKPOINT = '/kaggle/input/notebooks/dorast/fork-of-vae-b/runs/vae_domain_b_kl002_anneal/checkpoints/best.pt'  # adjust

for label, path in [('VAE1', VAE1_CHECKPOINT), ('VAE2', VAE2_CHECKPOINT)]:
    if os.path.exists(path):
        print(f'{label} checkpoint found: {path}')
    else:
        print(f'{label} checkpoint NOT found at {path} -- check the path above against !ls /kaggle/input/')


VAE1 checkpoint found: /kaggle/input/notebooks/dorast/vae-a/runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0300.pt
VAE2 checkpoint found: /kaggle/input/notebooks/dorast/fork-of-vae-b/runs/vae_domain_b_kl002_anneal/checkpoints/best.pt


## 5a. RECOMMENDED: attach your real old photos as a Kaggle Dataset

Same real-photo dataset used for VAE1 -- reuse the same attached dataset here rather than re-scraping.


In [14]:
REAL_PHOTO_DIR = '/kaggle/input/datasets/dorast/real-old-photos-vae1'  # adjust to your actual dataset slug

if os.path.isdir(REAL_PHOTO_DIR):
    n_images = len([f for f in os.listdir(os.path.join(REAL_PHOTO_DIR, 'images'))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Found {n_images} images at {REAL_PHOTO_DIR}')
else:
    print(f'{REAL_PHOTO_DIR} not found yet -- attach your dataset via Add Data first, '
          f'or use section 5b to scrape fresh instead.')


Found 2978 images at /kaggle/input/datasets/dorast/real-old-photos-vae1


## 5b. ALTERNATIVE: scrape fresh real old photos directly in this notebook

In [15]:
# %%writefile loc_scraper.py
# """
# Scrape real old/historical photographs from the Library of Congress's public
# loc.gov JSON API, for use as unpaired training data for VAE1 in the
# "Bringing Old Photos Back to Life" pipeline.

# No API key required (the loc.gov API is public), but it IS rate-limited --
# this script sleeps between requests to stay well under the limit.

# Docs: https://www.loc.gov/apis/json-and-yaml/

# Usage:
#     Single category (original behavior, unchanged):
#         python loc_scraper.py --query "portrait" --pages 20 --per-page 100 --out-dir ./real_old_photos

#     Multiple categories in one run, split by weight (new):
#         python loc_scraper.py \
#             --categories "portrait photograph,street scene,farm landscape,railroad,parade" \
#             --weights "0.2,0.2,0.2,0.2,0.2" \
#             --total-images 2000 \
#             --out-dir ./real_old_photos

#         --weights is optional -- omit it for an equal split across categories:
#         python loc_scraper.py --categories "street scene,farm landscape,railroad" --total-images 1500

# Output:
#     <out-dir>/images/*.jpg          -- downloaded photos
#     <out-dir>/manifest.csv          -- id, title, date, category, source url, local filename
#                                         (keep this for citation/provenance in your thesis)
# """

# import argparse
# import csv
# import io
# import os
# import time
# import sys
# from urllib.parse import urlencode

# import requests
# from PIL import Image, UnidentifiedImageError

# BASE_URL = "https://www.loc.gov/photos/"
# USER_AGENT = "thesis-research-scraper/1.0 (educational use; contact via loc.gov if issue)"


# def build_search_url(query, page, per_page, date_start=None, date_end=None):
#     params = {
#         "q": query,
#         "fo": "json",
#         "c": per_page,
#         "sp": page,
#         # restrict to items that are actually digitized/online so downloads succeed
#         "fa": "online-format:image",
#     }
#     if date_start and date_end:
#         params["dates"] = f"{date_start}/{date_end}"
#     return f"{BASE_URL}?{urlencode(params)}"


# def pick_image_url(item, max_pref_index=-2):
#     """
#     LOC's `image_url` field is a list of URLs at increasing resolution, with
#     the largest/highest-res last. Downloading the very largest can mean huge
#     TIFFs, so by default we take the second-to-last (max_pref_index=-2) which
#     is usually a large-but-reasonable JPEG. Falls back to the last available
#     entry if the list is shorter than expected.
#     """
#     urls = item.get("image_url")
#     if not urls:
#         return None
#     if len(urls) == 1:
#         return urls[0]
#     try:
#         return urls[max_pref_index]
#     except IndexError:
#         return urls[-1]


# def fetch_page(session, url, retries=3, backoff=5):
#     for attempt in range(retries):
#         try:
#             resp = session.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
#             if resp.status_code == 429:
#                 wait = backoff * (attempt + 1)
#                 print(f"  Rate limited (429). Waiting {wait}s...")
#                 time.sleep(wait)
#                 continue
#             resp.raise_for_status()
#             return resp.json()
#         except requests.RequestException as e:
#             print(f"  Request failed ({e}), retrying ({attempt + 1}/{retries})...")
#             time.sleep(backoff)
#     print(f"  Giving up on {url} after {retries} retries.")
#     return None


# def download_image(session, url, out_path, min_size=254, retries=3):
#     """
#     Downloads to memory first, validates it, and only then writes to disk.
#     Returns (success: bool, reason: str) so callers can log why something
#     was skipped.
#     """
#     for attempt in range(retries):
#         try:
#             resp = session.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
#             resp.raise_for_status()

#             content_type = resp.headers.get("Content-Type", "")
#             if not content_type.startswith("image"):
#                 return False, f"not an image (Content-Type: {content_type or 'unknown'})"

#             content = resp.content
#             if not content:
#                 return False, "empty response body"

#             # Validate it's a real, decodable image (catches truncated/corrupt downloads)
#             try:
#                 with Image.open(io.BytesIO(content)) as img:
#                     img.verify()
#                 # verify() invalidates the file pointer -- reopen to read dimensions
#                 with Image.open(io.BytesIO(content)) as img:
#                     width, height = img.size
#             except (UnidentifiedImageError, OSError) as e:
#                 return False, f"invalid/corrupt image ({e})"

#             if width <= min_size or height <= min_size:
#                 return False, f"too small ({width}x{height}, need > {min_size}x{min_size})"

#             with open(out_path, "wb") as f:
#                 f.write(content)
#             return True, "ok"

#         except requests.RequestException as e:
#             print(f"    Download failed ({e}), retrying ({attempt + 1}/{retries})...")
#             time.sleep(3)

#     return False, "download failed after retries"


# def format_duration(seconds):
#     seconds = int(seconds)
#     hours, remainder = divmod(seconds, 3600)
#     minutes, secs = divmod(remainder, 60)
#     if hours:
#         return f"{hours}h {minutes}m {secs}s"
#     if minutes:
#         return f"{minutes}m {secs}s"
#     return f"{secs}s"


# def run_category(session, query, target_count, images_dir, writer, manifest_file, args, start_time,
#                   overall_downloaded, overall_skipped, max_pages=None):
#     """
#     Scrapes a single category/query until either `target_count` images from
#     THIS category have been downloaded, or there are no more result pages.
#     Returns (category_downloaded, category_skipped, overall_downloaded, overall_skipped).

#     max_pages caps how many pages we'll try before giving up on a category
#     even if the target wasn't hit (protects against burning huge amounts of
#     time on a query that mostly returns skippable/too-small/non-image results).
#     """
#     if max_pages is None:
#         # generous cap: enough pages to plausibly reach target even with a
#         # high skip rate, but not unbounded
#         max_pages = max(10, (target_count // args.per_page + 1) * 4)

#     category_downloaded = 0
#     category_skipped = 0

#     for page in range(1, max_pages + 1):
#         if category_downloaded >= target_count:
#             break

#         url = build_search_url(query, page, args.per_page, args.date_start, args.date_end)
#         print(f"  [{query}] Fetching page {page}: {url}")
#         data = fetch_page(session, url)
#         time.sleep(args.sleep)

#         if not data:
#             print(f"  [{query}] No data returned, stopping this category.")
#             break

#         results = data.get("results", [])
#         if not results:
#             print(f"  [{query}] No results on this page, stopping this category.")
#             break

#         for item in results:
#             if category_downloaded >= target_count:
#                 break

#             item_id = item.get("id", "unknown").rstrip("/").split("/")[-1]
#             title = (item.get("title") or "untitled").strip()
#             date = item.get("date", "")
#             image_url = pick_image_url(item)
#             source_page = item.get("id", "")

#             if not image_url:
#                 category_skipped += 1
#                 overall_skipped += 1
#                 continue

#             local_filename = f"{item_id}.jpg"
#             local_path = os.path.join(images_dir, local_filename)

#             if os.path.exists(local_path):
#                 category_skipped += 1
#                 overall_skipped += 1
#                 continue

#             ok, reason = download_image(session, image_url, local_path, min_size=args.min_size)
#             if ok:
#                 writer.writerow([item_id, title, date, query, source_page, image_url, local_filename])
#                 manifest_file.flush()
#                 category_downloaded += 1
#                 overall_downloaded += 1
#                 if overall_downloaded % 25 == 0:
#                     elapsed = time.time() - start_time
#                     print(f"  Downloaded {overall_downloaded} images total so far... "
#                           f"(elapsed: {format_duration(elapsed)})")
#             else:
#                 category_skipped += 1
#                 overall_skipped += 1
#                 print(f"    [{query}] Skipped {item_id}: {reason}")

#             time.sleep(args.download_sleep)

#         pagination = data.get("pagination", {})
#         if not pagination.get("next"):
#             print(f"  [{query}] Reached last page of results.")
#             break

#     print(f"  [{query}] done: {category_downloaded}/{target_count} target images "
#           f"({category_skipped} skipped)")
#     return category_downloaded, category_skipped, overall_downloaded, overall_skipped


# def compute_category_targets(categories, weights, total_images):
#     """
#     Splits total_images across categories according to weights (normalized
#     automatically, so they don't need to sum to exactly 1.0). Rounding is
#     corrected on the last category so the targets sum exactly to total_images.
#     """
#     weight_sum = sum(weights)
#     normalized = [w / weight_sum for w in weights]
#     targets = [round(total_images * w) for w in normalized[:-1]]
#     targets.append(total_images - sum(targets))  # last one absorbs rounding error
#     return dict(zip(categories, targets))


# def main():
#     parser = argparse.ArgumentParser(description="Scrape real old photos from the Library of Congress API.")
#     parser.add_argument("--query", type=str, default="portrait photograph",
#                          help="single search term (ignored if --categories is set), e.g. 'family photograph'")
#     parser.add_argument("--categories", type=str, default=None,
#                          help="comma-separated list of search terms to split scraping across in one run, "
#                               "e.g. 'portrait,street scene,farm landscape,railroad,parade'. "
#                               "Overrides --query when set.")
#     parser.add_argument("--weights", type=str, default=None,
#                          help="comma-separated weights matching --categories, e.g. '0.3,0.2,0.2,0.15,0.15'. "
#                               "Don't need to sum to 1 (auto-normalized). Omit for an equal split.")
#     parser.add_argument("--total-images", type=int, default=1000,
#                          help="total images to collect across all categories (only used with --categories)")
#     parser.add_argument("--start-page", type=int, default=1,
#                          help="page number to start from (use this to continue a previous run, "
#                               "e.g. --start-page 21 to continue after a run that covered pages 1-20)")
#     parser.add_argument("--pages", type=int, default=10, help="number of result pages to fetch, starting from --start-page")
#     parser.add_argument("--per-page", type=int, default=100, choices=[25, 50, 100, 150],
#                          help="results per page (LOC API allows 25/50/100/150)")
#     parser.add_argument("--min-size", type=int, default=254,
#                          help="minimum width AND height in pixels; images not strictly larger than this in both "
#                               "dimensions are discarded")
#     parser.add_argument("--date-start", type=str, default=None, help="e.g. 1850")
#     parser.add_argument("--date-end", type=str, default=None, help="e.g. 1970")
#     parser.add_argument("--out-dir", type=str, default="./real_old_photos")
#     parser.add_argument("--sleep", type=float, default=3.0,
#                          help="seconds to sleep between API requests (be polite -- LOC enforces rate limits)")
#     parser.add_argument("--download-sleep", type=float, default=0.5,
#                          help="seconds to sleep between image downloads")
#     args = parser.parse_args()

#     start_time = time.time()

#     images_dir = os.path.join(args.out_dir, "images")
#     os.makedirs(images_dir, exist_ok=True)
#     manifest_path = os.path.join(args.out_dir, "manifest.csv")

#     session = requests.Session()
#     downloaded = 0
#     skipped = 0

#     write_header = not os.path.exists(manifest_path)
#     manifest_file = open(manifest_path, "a", newline="", encoding="utf-8")
#     writer = csv.writer(manifest_file)
#     if write_header:
#         writer.writerow(["id", "title", "date", "category", "source_page_url", "image_url", "local_filename"])
#     elif args.categories:
#         # Warn if we're appending category-tagged rows to a manifest that
#         # predates the category column -- the CSV will have mixed row shapes.
#         with open(manifest_path, "r", encoding="utf-8") as f:
#             existing_header = f.readline().strip()
#         if "category" not in existing_header:
#             print(f"Warning: {manifest_path} already exists without a 'category' column. "
#                   f"New rows will have an extra column, producing a mixed-format CSV. "
#                   f"Consider using a fresh --out-dir if you want a clean manifest.")

#     downloaded = 0
#     skipped = 0

#     if args.categories:
#         categories = [c.strip() for c in args.categories.split(",") if c.strip()]
#         if args.weights:
#             weights = [float(w.strip()) for w in args.weights.split(",")]
#             if len(weights) != len(categories):
#                 raise ValueError(f"--weights has {len(weights)} values but --categories has "
#                                   f"{len(categories)} -- they must match.")
#         else:
#             weights = [1.0] * len(categories)  # equal split

#         targets = compute_category_targets(categories, weights, args.total_images)
#         print("Category targets for this run:")
#         for cat, target in targets.items():
#             print(f"  {cat!r}: {target} images")
#         print()

#         for cat, target in targets.items():
#             if target <= 0:
#                 continue
#             cat_downloaded, cat_skipped, downloaded, skipped = run_category(
#                 session, cat, target, images_dir, writer, manifest_file, args,
#                 start_time, downloaded, skipped,
#             )
#             elapsed = time.time() - start_time
#             print(f"Running total: {downloaded} downloaded, {skipped} skipped. "
#                   f"Elapsed: {format_duration(elapsed)}\n")

#     else:
#         # Original single-query behavior, preserved exactly, just with a
#         # 'category' column added to the manifest for consistency.
#         end_page = args.start_page + args.pages - 1
#         for page in range(args.start_page, end_page + 1):
#             url = build_search_url(args.query, page, args.per_page, args.date_start, args.date_end)
#             print(f"Fetching page {page}/{end_page}: {url}")
#             data = fetch_page(session, url)
#             time.sleep(args.sleep)

#             if not data:
#                 print("  No data returned, stopping.")
#                 break

#             results = data.get("results", [])
#             if not results:
#                 print("  No results on this page, stopping (may have reached the end).")
#                 break

#             for item in results:
#                 item_id = item.get("id", "unknown").rstrip("/").split("/")[-1]
#                 title = (item.get("title") or "untitled").strip()
#                 date = item.get("date", "")
#                 image_url = pick_image_url(item)
#                 source_page = item.get("id", "")

#                 if not image_url:
#                     skipped += 1
#                     continue

#                 local_filename = f"{item_id}.jpg"
#                 local_path = os.path.join(images_dir, local_filename)

#                 if os.path.exists(local_path):
#                     skipped += 1
#                     continue

#                 ok, reason = download_image(session, image_url, local_path, min_size=args.min_size)
#                 if ok:
#                     writer.writerow([item_id, title, date, args.query, source_page, image_url, local_filename])
#                     manifest_file.flush()
#                     downloaded += 1
#                     if downloaded % 25 == 0:
#                         elapsed = time.time() - start_time
#                         print(f"  Downloaded {downloaded} images so far... (elapsed: {format_duration(elapsed)})")
#                 else:
#                     skipped += 1
#                     print(f"    Skipped {item_id}: {reason}")

#                 time.sleep(args.download_sleep)

#             pagination = data.get("pagination", {})
#             elapsed = time.time() - start_time
#             print(f"  Page {page} done. Total so far: {downloaded} downloaded, {skipped} skipped. "
#                   f"Elapsed: {format_duration(elapsed)}")
#             if not pagination.get("next"):
#                 print("  Reached last page of results.")
#                 break

#     manifest_file.close()
#     total_elapsed = time.time() - start_time
#     rate = downloaded / total_elapsed * 60 if total_elapsed > 0 else 0
#     print(f"\nDone. Downloaded {downloaded} images, skipped {skipped}.")
#     print(f"Total runtime: {format_duration(total_elapsed)} ({rate:.1f} images/min)")
#     print(f"Images saved to: {images_dir}")
#     print(f"Manifest (for citation/provenance) saved to: {manifest_path}")


# if __name__ == "__main__":
#     main()


In [16]:
# Example: weighted multi-category scrape, matching the balancing approach
# from earlier in the project. Adjust categories/weights/total as needed.
# REAL_PHOTO_DIR = '/kaggle/working/real_old_photos'
# result = subprocess.run([
#     'python', 'loc_scraper.py',
#     '--categories', 'portrait photograph,street scene,farm landscape,railroad,parade',
#     '--weights', '0.15,0.25,0.2,0.2,0.2',
#     '--total-images', '3000',
#     '--out-dir', REAL_PHOTO_DIR,
# ])
# result.check_returncode()


## 6. Get VOC2012 clean images — automatic, no manual download

In [17]:
import torchvision.datasets as tvds

VOC_ROOT = '/kaggle/working/voc_data'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
    print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
else:
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


100%|██████████| 2.00G/2.00G [00:49<00:00, 40.3MB/s]


VOC2012 ready: 17125 images at /kaggle/working/voc_data/VOCdevkit/VOC2012/JPEGImages


## 6.5. Build validation subsets (real photos AND clean images)

Separate slices for `--val-real-photo-dir` and `--val-clean-dir` -- disjoint from what training uses.


In [18]:
import random, shutil

VAL_REAL_PHOTO_DIR = '/kaggle/working/real_photos_val'
os.makedirs(os.path.join(VAL_REAL_PHOTO_DIR, 'images'), exist_ok=True)
existing_val_real = [f for f in os.listdir(os.path.join(VAL_REAL_PHOTO_DIR, 'images'))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
TARGET_N_VAL_REAL = 30

if len(existing_val_real) >= TARGET_N_VAL_REAL:
    print(f'{len(existing_val_real)} real-photo validation images already present, skipping.')
else:
    images_dir = os.path.join(REAL_PHOTO_DIR, 'images')
    all_real = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42)
    val_real_subset = random.sample(all_real, min(TARGET_N_VAL_REAL, len(all_real)))
    for fname in val_real_subset:
        shutil.copy(os.path.join(images_dir, fname), os.path.join(VAL_REAL_PHOTO_DIR, 'images', fname))
    print(f'Copied {len(val_real_subset)} real-photo validation images')

VAL_SUBSET_DIR = '/kaggle/working/voc_val_subset/images'
os.makedirs(VAL_SUBSET_DIR, exist_ok=True)
existing_val_clean = [f for f in os.listdir(VAL_SUBSET_DIR) if f.lower().endswith('.jpg')]
TARGET_N_VAL_CLEAN = 150

if len(existing_val_clean) >= TARGET_N_VAL_CLEAN:
    print(f'{len(existing_val_clean)} clean validation images already present, skipping.')
else:
    all_voc_images = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
    random.seed(43)
    val_clean_subset = random.sample(all_voc_images, TARGET_N_VAL_CLEAN)
    for fname in val_clean_subset:
        shutil.copy(os.path.join(VOC_JPEG_DIR, fname), os.path.join(VAL_SUBSET_DIR, fname))
    print(f'Copied {len(val_clean_subset)} clean validation images')


Copied 30 real-photo validation images
Copied 150 clean validation images


## 7. Generate damage masks

Four damage types, each into its OWN folder, at the validated intensity settings.


In [19]:
import os

MASKS_DIR = '/kaggle/input/datasets/dorast/generated-masks'

DAMAGE_TYPES = ['dirt', 'scratches', 'smut', 'spots']  # adjust if the ls output above shows different names

MASKS_DIRS = [os.path.join(MASKS_DIR, t) for t in DAMAGE_TYPES]

for d in MASKS_DIRS:
    n = len([f for f in os.listdir(d) if f.startswith('mask_')]) if os.path.isdir(d) else 0
    print(f'{d}: {n} masks')

/kaggle/input/datasets/dorast/generated-masks/dirt: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/scratches: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/smut: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/spots: 3000 masks


## 8. Quick smoke test (small config, low resolution, WITH validation)

In [20]:
# import subprocess

# result = subprocess.run([
#     'python', 'train_translation_net.py',
#     '--vae1-checkpoint', VAE1_CHECKPOINT, '--vae2-checkpoint', VAE2_CHECKPOINT,
#     '--real-photo-dir', REAL_PHOTO_DIR, '--clean-dir', VOC_JPEG_DIR, '--masks-dir', *MASKS_DIRS,
#     '--val-real-photo-dir', VAL_REAL_PHOTO_DIR, '--val-clean-dir', VAL_SUBSET_DIR, '--val-masks-dir', *MASKS_DIRS,
#     '--epochs', '3', '--batch-size', '8', '--image-size', '64', '--num-workers', '2',
#     '--steps-per-epoch', '10', '--damage-weight', '5.0', '--val-every', '1',
#     '--log-every', '5', '--sample-every', '10', '--save-every', '1',
#     '--out-dir', './runs/translation_smoke_test', '--device', 'cuda',
# ])
# result.check_returncode()


## 9. View a translation sample

Compares synthetic (has ground truth) and real (no ground truth, qualitative only) translation pathways.


In [21]:
# import glob
# from PIL import Image
# import matplotlib.pyplot as plt

# sample_files = sorted(glob.glob('runs/translation_smoke_test/samples/*.png'))
# if sample_files:
#     img = Image.open(sample_files[-1])
#     plt.figure(figsize=(14, 10))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.title(f'Latest sample: {sample_files[-1]}')
#     plt.show()
# else:
#     print('No samples found yet — check the training cell above ran successfully.')


## 10. Baseline sanity check: short run WITH validation

Real architecture size, `--image-size 128`.


In [22]:
import subprocess
result = subprocess.run([
    'python', 'train_translation_net.py',
    '--vae1-checkpoint', VAE1_CHECKPOINT, '--vae2-checkpoint', VAE2_CHECKPOINT,
    '--real-photo-dir', REAL_PHOTO_DIR, '--clean-dir', VOC_JPEG_DIR, '--masks-dir', *MASKS_DIRS,
    '--val-real-photo-dir', VAL_REAL_PHOTO_DIR, '--val-clean-dir', VAL_SUBSET_DIR, '--val-masks-dir', *MASKS_DIRS,
    '--epochs', '550', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
    '--steps-per-epoch', '100',
    '--damage-weight', '5.0', '--val-every', '1',
    '--image-adv-weight', '0.0003',
    '--image-disc-lr', '2e-5',
    '--log-every', '20', '--sample-every', '100', '--save-every', '5',
    '--out-dir', './runs/spectral_norm_test', '--device', 'cuda',
    '--resume', '/kaggle/input/notebooks/dorast/fork-of-fork-of-translation-net-kaggle/runs/spectral_norm_test/checkpoints/translation_net_epoch0400.pt',
])
result.check_returncode()

[2026-09-14 06:18:00] Using device: cuda
[2026-09-14 06:18:00]   GPU: Tesla T4
[2026-09-14 06:18:00] Loading frozen VAE1 from /kaggle/input/notebooks/dorast/vae-a/runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0300.pt
[2026-09-14 06:18:04] Loading frozen VAE2 from /kaggle/input/notebooks/dorast/fork-of-vae-b/runs/vae_domain_b_kl002_anneal/checkpoints/best.pt
[2026-09-14 06:18:07] Building datasets...
[2026-09-14 06:18:07] Real photos: 2978. Synthetic pairs: 17125 clean images, 12000 masks.
[2026-09-14 06:18:07] Fetching fixed sample batches for visualization...
[2026-09-14 06:18:08] Datasets ready.
[2026-09-14 06:18:08] Building validation dataset from /kaggle/working/voc_val_subset/images / ['/kaggle/input/datasets/dorast/generated-masks/dirt', '/kaggle/input/datasets/dorast/generated-masks/scratches', '/kaggle/input/datasets/dorast/generated-masks/smut', '/kaggle/input/datasets/dorast/generated-masks/spots']...
[2026-09-14 06:18:08] Loaded 150 validation samples (--val-real-photo

Epoch 401/550:   0%|          | 0/100 [00:09<?, ?batch/s, sup=0.0609, adv_d=0.6417]

[2026-09-14 06:18:18]   step 40020: data_time=0.000s compute_time=0.367s


Epoch 401/550:  21%|██        | 21/100 [00:17<00:37,  2.09batch/s, sup=0.0592, adv_d=0.4983]

[2026-09-14 06:18:26]   step 40040: data_time=0.000s compute_time=0.373s


...

Epoch 550/550:  92%|█████████▏| 92/100 [00:44<00:03,  2.24batch/s, sup=0.0529, adv_d=0.6057]

[2026-09-14 08:12:35]   step 55000: data_time=0.000s compute_time=0.446s


Epoch 550/550:  92%|█████████▏| 92/100 [00:44<00:03,  2.24batch/s, sup=0.0579, adv_d=0.5927]

[2026-09-14 08:12:35]   Saved sample grids: ./runs/spectral_norm_test/samples/step_0055000_synthetic.png / ./runs/spectral_norm_test/samples/step_0055000_real.png
[2026-09-14 08:12:35] [Epoch 550/550] sup=0.0581 identity=0.0464 adv_g=1.6365 adv_d=0.5871 adv_g_img=11.3460 adv_d_img=0.0000 epoch_time=44s total_elapsed=1h 54m 26s


[2026-09-14 08:12:36]   [Validation] epoch 550: weighted_supervised_loss=0.0795 plain_l1=0.0556
[2026-09-14 08:12:36]   Saved checkpoint: ./runs/spectral_norm_test/checkpoints/translation_net_epoch0550.pt
[2026-09-14 08:12:36] Training complete. Total time: 1h 54m 27s


## 11. Full training run

In [23]:
# Example full run:
# result = subprocess.run([
#     'python', 'train_translation_net.py',
#     '--vae1-checkpoint', VAE1_CHECKPOINT, '--vae2-checkpoint', VAE2_CHECKPOINT,
#     '--real-photo-dir', REAL_PHOTO_DIR, '--clean-dir', VOC_JPEG_DIR, '--masks-dir', *MASKS_DIRS,
#     '--val-real-photo-dir', VAL_REAL_PHOTO_DIR, '--val-clean-dir', VAL_SUBSET_DIR, '--val-masks-dir', *MASKS_DIRS,
#     '--epochs', '50', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
#     '--damage-weight', '5.0', '--val-every', '1',
#     '--out-dir', './runs/translation_net', '--device', 'cuda',
# ])
# result.check_returncode()

# To resume from a previous session's checkpoint:
# result = subprocess.run([
#     'python', 'train_translation_net.py',
#     '--vae1-checkpoint', VAE1_CHECKPOINT, '--vae2-checkpoint', VAE2_CHECKPOINT,
#     '--real-photo-dir', REAL_PHOTO_DIR, '--clean-dir', VOC_JPEG_DIR, '--masks-dir', *MASKS_DIRS,
#     '--val-real-photo-dir', VAL_REAL_PHOTO_DIR, '--val-clean-dir', VAL_SUBSET_DIR, '--val-masks-dir', *MASKS_DIRS,
#     '--epochs', '50', '--batch-size', '16', '--image-size', '128',
#     '--damage-weight', '5.0', '--val-every', '1',
#     '--out-dir', './runs/translation_net', '--device', 'cuda',
#     '--resume', '/kaggle/working/runs/translation_net/checkpoints/<latest>.pt',
# ])
# result.check_returncode()


In [24]:
import shutil

for folder in ['/kaggle/working/real_photos_val']:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
        print(f'Deleted: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

Deleted: /kaggle/working/real_photos_val


In [25]:
import shutil

for folder in ['/kaggle/working/voc_data']:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
        print(f'Deleted: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

Deleted: /kaggle/working/voc_data
